# Marco 4 — Treino Multi-Seed (5 Sementes) + Reamostragem L-BFGS em GPU (Colab T4)

**Instruções (1 clique):**
1. No menu superior do Google Colab, vá em **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4**.
2. Clique em **Ambiente de execução → Executar tudo** (`Ctrl+F9`).
3. Ao final (~10–15 min na T4), o navegador baixará automaticamente `resultados_colab_marco4_5seeds.zip` com os 5 checkpoints `.pt`, os JSONs de avaliação e o resumo estatístico.

In [ ]:
!nvidia-smi
import os, sys, time, json, math, zipfile
import numpy as np
import torch

assert torch.cuda.is_available(), 'Ative a GPU T4 em Ambiente de execucao -> Alterar tipo de ambiente de execucao!'
print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.get_device_name(0))

if not os.path.exists('/content/mestrado'):
    !git clone --depth 1 https://github.com/dafnhe-thor-barros/mestrado.git /content/mestrado
else:
    !cd /content/mestrado && git pull --ff-only

WORKDIR = '/content/mestrado/simulacoes/v2_auditado'
os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)
import modelo_pinn as mp
print('Modulo carregado:', mp.__file__)

In [ ]:
# Wrapper GPU: executa EXATAMENTE as mesmas equacoes, perdas em K-equivalente e amostragem de modelo_pinn.py (v2) em CUDA
DEV = torch.device('cuda')

def to_dev_batch(b, dev):
    return {k: (v.to(dev) if torch.is_tensor(v) else v) for k, v in b.items()}

def train_gpu(epochs=20000, ns=48, lr=1e-3, seed=0, lbfgs_steps=50, ns_lbfgs=96, lbfgs_seed_offset=1000, init=None, out='v2c_gpu.pt'):
    torch.manual_seed(seed)
    np.random.seed(seed)
    gen = torch.Generator().manual_seed(seed)
    ph = mp.Phys()
    # Move LO e HI do modulo para GPU durante o treino
    mp.LO = mp.LO.to(DEV)
    mp.HI = mp.HI.to(DEV)
    if init:
        ck = torch.load(init, map_location='cpu')
        model = mp.PINN(ck['width'], ck['depth'], ck['hard_ic']).to(DEV)
        model.load_state_dict(ck['state'])
    else:
        model = mp.PINN(64, 4, True).to(DEV)

    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs, 1), eta_min=lr * 0.01)
    hist = []
    t0 = time.perf_counter()
    for it in range(1, epochs + 1):
        opt.zero_grad()
        b = to_dev_batch(mp.make_batch(ns, gen, sampling='corr'), DEV)
        parts = mp.losses_from_batch(model, ph, b)
        loss = mp.total(parts)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
        opt.step()
        sched.step()
        if it % 5000 == 0 or it == 1:
            rec = mp._rec('adam', it, parts, loss)
            hist.append(rec)
            print(f"[seed={seed}] adam it={it:5d} ({time.perf_counter()-t0:.1f}s) loss={rec['loss']:.4f} rms_K=({rec['rms_K_pde']:.2f}, {rec['rms_K_bc']:.2f}, {rec['rms_K_mod']:.2f})", flush=True)

    if lbfgs_steps > 0:
        fixed = to_dev_batch(mp.make_batch(ns_lbfgs, torch.Generator().manual_seed(seed + lbfgs_seed_offset), sampling='corr'), DEV)
        mp.lbfgs_phase(model, ph, fixed, lbfgs_steps, mp.W_DEF, log_every=25, hist=hist)

    # Volta para CPU para salvar checkpoint 100% compativel com o notebook local
    model_cpu = model.to('cpu')
    mp.LO = mp.LO.to('cpu')
    mp.HI = mp.HI.to('cpu')
    torch.save({'state': model_cpu.state_dict(), 'width': 64, 'depth': 4, 'hard_ic': True, 'seed': seed, 'version': 2, 'hist': hist}, out)
    print(f'Salvo {out} em {time.perf_counter()-t0:.1f}s')
    return model_cpu

In [ ]:
# 1) Campanha de 5 sementes independentes (Adam 20k + L-BFGS 50, ns_lbfgs=96)
OUT_DIR = '/content/resultados_marco4'
os.makedirs(OUT_DIR, exist_ok=True)

seeds_summary = []
for s in [0, 1, 2, 3, 4]:
    ckpt = os.path.join(OUT_DIR, f'v2c_seed{s}.pt')
    m = train_gpu(epochs=20000, seed=s, lbfgs_steps=50, ns_lbfgs=96, out=ckpt)
    m.eval()
    ev = mp.evaluate(m, ncases=20)
    ev_red = {k: ev[k] for k in ('losses_validacao_K2', 'losses_validacao_rms_K', 'resumo', 'custo')} | {
        'casos': {k: v for k, v in ev['casos'].items() if k in ('A_sol_quente', 'B_degrau_frio', 'R00', 'R04')}
    }
    with open(os.path.join(OUT_DIR, f'eval_v2c_seed{s}.json'), 'w', encoding='utf-8') as f:
        json.dump(ev_red, f, indent=1, ensure_ascii=False)
    nd = ev['resumo']['no_dominio_treino_v2']
    seeds_summary.append({
        'seed': s,
        'pior_in_domain_C': nd['pior_max_abs_C'],
        'mediana_in_domain_C': nd['mediana_max_abs_C'],
        'n_le_0p5C': nd['n_le_0p5C'],
        'pior_subest_pico_C': nd['pior_subestimacao_pico_Tj_C'],
        'rms_K_val_pde': ev['losses_validacao_rms_K']['L_pde'],
        'rms_K_val_bc': ev['losses_validacao_rms_K']['L_bc'],
    })

# 2) Reamostragem do lote de colocacao do L-BFGS a partir de v2b.pt (ns_lbfgs=96 e 192)
lbfgs_resamp = []
if os.path.exists('v2b.pt'):
    for ns_lb, off in [(96, 1000), (96, 2000), (192, 3000)]:
        ckpt = os.path.join(OUT_DIR, f'v2c_resamp_ns{ns_lb}_off{off}.pt')
        m = train_gpu(epochs=0, seed=0, lbfgs_steps=50, ns_lbfgs=ns_lb, lbfgs_seed_offset=off, init='v2b.pt', out=ckpt)
        m.eval()
        ev = mp.evaluate(m, ncases=20)
        nd = ev['resumo']['no_dominio_treino_v2']
        lbfgs_resamp.append({
            'ns_lbfgs': ns_lb, 'offset': off,
            'pior_in_domain_C': nd['pior_max_abs_C'],
            'mediana_in_domain_C': nd['mediana_max_abs_C'],
            'n_le_0p5C': nd['n_le_0p5C'],
            'pior_subest_pico_C': nd['pior_subestimacao_pico_Tj_C'],
            'rms_K_val_pde': ev['losses_validacao_rms_K']['L_pde'],
        })

pior_arr = [r['pior_in_domain_C'] for r in seeds_summary]
med_arr = [r['mediana_in_domain_C'] for r in seeds_summary]
sub_arr = [r['pior_subest_pico_C'] for r in seeds_summary]
final_summary = {
    'seeds_5': seeds_summary,
    'estatisticas_5_sementes': {
        'pior_in_domain_media_C': float(np.mean(pior_arr)), 'pior_in_domain_std_C': float(np.std(pior_arr)), 'pior_entre_sementes_C': float(np.max(pior_arr)),
        'mediana_in_domain_media_C': float(np.mean(med_arr)), 'mediana_in_domain_std_C': float(np.std(med_arr)),
        'pior_subestimacao_entre_sementes_C': float(np.min(sub_arr)),
    },
    'reamostragem_lbfgs': lbfgs_resamp,
}
sum_path = os.path.join(OUT_DIR, 'saida_marco4_colab_5seeds_resumo.json')
with open(sum_path, 'w', encoding='utf-8') as f:
    json.dump(final_summary, f, indent=1, ensure_ascii=False)
print(json.dumps(final_summary, indent=1, ensure_ascii=False))

In [ ]:
# Compacta tudo e baixa automaticamente
zip_path = '/content/resultados_colab_marco4_5seeds.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fn in sorted(os.listdir(OUT_DIR)):
        zf.write(os.path.join(OUT_DIR, fn), arcname=fn)
print('ZIP gerado:', zip_path, os.path.getsize(zip_path), 'bytes')
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print('Baixe manualmente em:', zip_path, e)